# Biblioteka Bokeh

Bokeh to biblioteka wyróżniająca się wysokim poziomem interaktywności. Umożliwia dostosowywanie wizualizacji w czasie rzeczywistym  dla użytkowników, którzy nie mają styczności z kodem.

In [ ]:
from bokeh.io import output_notebook
from bokeh.layouts import column, row
from bokeh.layouts import gridplot
from bokeh.models import ColorPicker, Spinner
from bokeh.models import FactorRange, CrosshairTool
from bokeh.models import LassoSelectTool, PolySelectTool, WheelZoomTool, UndoTool, ResetTool, RedoTool, SaveTool, \
    HoverTool, Span, PanTool
from bokeh.models import RangeSlider, CustomJS, Range1d
from bokeh.plotting import figure, show

In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

athlete_events = pd.read_csv('athlete_events.csv', sep=';')

--------------------

##### ⭐ Zadanie 1: 

Przygotuj wykres punktowy (`circle`) opierając się na danych, które sam wybierzesz w sensowny sposób. W rozwiązaniu zdefiniuj własny pasek narzędzi składający się z narzędzi:

- `LassoSelectTool` i `xPanTool` z kategorii narzędzi *Gestures* (*Pan/Drag tools*),
- `PolySelectTool` z kategorii narzędzi *Gestures* (*Click/Tap tools*),
- `WheelZoomTool` z kategorii narzędzi *Gestures* (*Scroll/Pinch tools*),
- `UndoTool`, `RedoTool`, `ResetTool` i `SaveTool` z kategorii narzędzi *Actions*,
- `HoverTool` z etykietami danych z kategorii narzędzi *Inspectors*.

Zapoznaj się z parametrem `toolbar_location` i dobierz dla niego nową wartość. Zapoznaj się z metodą `autohide` obiektu `toolbar` i dobierz dla niego taką wartość, żeby pasek narzędzi chował się po zjechaniu kursorem myszy z wizualizacji. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
scatter_data = athlete_events.loc[
    (athlete_events['Sport'] == 'Taekwondo') &
    (athlete_events['Sex'] == 'F')
    , ['Weight', 'Height']
].drop_duplicates().dropna()

output_notebook()

hover = HoverTool(
    description='Najedź',
    tooltips=[
        ('Waga', '@x kg'),
        ('Wzrost', '@y cm')
    ]
)

p = figure(
    title='Waga i wzrost zawodniczek taekwondo',
    width=600,
    height=600,
    x_axis_label='Waga (kg)',
    y_axis_label='Wzrost (cm)',
    toolbar_location='below',
    y_range=(145, 195),
    x_range=(40, 90),
    tools=[LassoSelectTool(description='Zaznacz punkty (Lasso)'), WheelZoomTool(description='Przybliż'), ResetTool(description='Resetuj'), PolySelectTool(description='Zaznacz punkty (Wielokąt)'), UndoTool(description='Powrót'), RedoTool(description='Powtórz'), PanTool(dimensions='width', description='Przesuń oś X'), SaveTool(description='Zapisz'), hover],
)

p.circle(scatter_data['Weight'], scatter_data['Height'], size=10, color='navy', line_color='red', alpha=0.75)
p.toolbar.autohide = True
p.title.align = 'center'

show(p)

##### ⭐ Zadanie 2:

Przygotuj wykres kolumnowy (`vbar`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Każda seria danych musi być przedstawiona na osobnym podwykresie (`subplot`) jednego obrazu. Zapoznaj się z metodami `column`, `row` i `gridplot` oraz dobierz dla nich właściwą wartość mając na uwadze wizualizację określonej liczby serii danych na osobnych podwykresach. Dodaj powiązane zachowania pomiędzy podwykresami:

- współdzielony ruchomy zakres dla wartości na osi X, także przy poziomym przesuwaniu wykresu,
- współdzielony wskaźnik będący narzędziem `CrosshairTool` widoczny jednocześnie na wszystkich podwykresach.

Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i ewentualnie legenda). 

In [ ]:
medal_types = ['Gold', 'Silver', 'Bronze']
countries = ['POL', 'TUR', 'UKR', 'SWE']
pol_countries = ['Polskę', 'Turcję', 'Ukrainę', 'Szwecję']

vbar_data = (athlete_events.loc[
    (athlete_events['Season'] == 'Summer') &
    (athlete_events['NOC'].isin(countries)) &
    (athlete_events['Year'] >= 1994) &
    (athlete_events['Medal'].notna()), 
    ['NOC', 'Year', 'Event']
]
             .drop_duplicates()
             .groupby(['NOC', 'Year'])
             .size()
             .reset_index(name='Count')
             )

vbar_data['Year'] = vbar_data['Year'].astype(str)

width = Span(dimension='width', line_width=2, line_color='red')
height = Span(dimension='height', line_width=2, line_color='red')
crosshair = CrosshairTool(overlay=[width, height])
pan_tool = PanTool(dimensions='width')

shared_x_range = FactorRange(factors=vbar_data['Year'].unique())

figures = [
    figure(
        title=f'Liczba zdobytych medali przez {c}',
        width=250,
        height=250,
        x_axis_label='Rok' if i <= 1 else '',
        y_axis_label='Liczba konkurencji' if i % 2 == 0 else '',
        toolbar_location='below',
        tools=[crosshair, pan_tool],
        x_range=shared_x_range,
    ) for i, c in enumerate(pol_countries)
]

for i, code in enumerate(countries):
    tmp = vbar_data[vbar_data['NOC'] == code]
    
    figures[i].vbar(
        x=tmp['Year'],
        top=tmp['Count'],
        width=0.7,
        color='#1f77b4'
    )
    figures[i].title.text_color = '#333333'
    figures[i].title.align = 'center'
    figures[i].background_fill_color = '#f5f5f5'
    figures[i].background_fill_alpha = 0.8
    figures[i].grid.grid_line_color = '#dddddd'
    figures[i].grid.grid_line_alpha = 0.7
    figures[i].legend.visible = False

grid = gridplot([[figures[2], figures[3]], [figures[0], figures[1]]], width=600, height=600)
show(grid)

##### ⭐ Zadanie 3:

Przygotuj wykres liniowy (`line`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z parametrem `location` obiektu `legend` i dobierz dla niego nową wartość. Zapoznaj się z parametrem `click_policy` obiektu `legend` i dobierz dla niego taką wartość, żeby po kliknięciu w tytuł danej serii danych w legendzie, jej wizualizacji znikała z wykresu. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i legenda). 

In [ ]:
swim = r"^Swimming Men's (100|200|400|1,500) metres Freestyle$"
line_data = athlete_events.loc[
    (athlete_events['Sport'] == 'Swimming') & 
    (athlete_events['Sex'] == 'M') & 
    (athlete_events['Event'].str.contains(swim, regex=True)) & 
    (athlete_events['Year'] >= 1952) & 
    (athlete_events['Medal'].notna()), 
    ['Year', 'Event', 'Height']
].groupby(['Year', 'Event']).mean().reset_index()

p = figure(
    title='Średni wzrost medalistów w pływaniu',
    width=800,
    height=600,
    x_axis_label='Rok',
    y_axis_label='Wzrost (cm)',
    x_range=(1948, 2020)
)

event_names = {event_name: f'{event_name.split(' ')[2]} metrów' for event_name in line_data['Event'].unique()}
colors = ['blue', 'green', 'orange', 'red']

for i, (key, value) in enumerate(event_names.items()):
    tmp = line_data[line_data['Event'] == key]
    p.line(tmp['Year'], tmp['Height'], line_width=2, color=colors[i], legend_label=value)
    p.scatter(tmp['Year'], tmp['Height'], size=6, color=colors[i], alpha=0.7, legend_label=value)

p.legend.location = 'top_left'
p.legend.title = 'Dystans'
p.legend.click_policy = 'hide'
p.title.text_color = '#333333'
p.title.align = 'center'
p.background_fill_color = '#f5f5f5'
p.background_fill_alpha = 0.8
p.grid.grid_line_color = '#dddddd'
p.grid.grid_line_alpha = 0.7

output_notebook()
show(p)

##### ⭐ Zadanie 4:

Przedstaw na dowolnym wykresie dowolne zestawienie danych przygotowane na podstawie pliku `athlete_events.csv` z pierwszego tygodnia. Zadbaj o czytelność wykresu. Do swojej wizualizacji dodaj co najmniej 3 różne widgety, które będą miały na nią wpływ:

- `AutocompleteInput`,
- `Button`,
- `CheckboxButtonGroup`,
- `CheckboxGroup`,
- `ColorPicker`,
- `DataCube`,
- `DataTable`,
- `DatePicker`,
- `DateRangePicker`,
- `MultipleDatePicker`,
- `DatetimePicker`,
- `DatetimeRangePicker`,
- `MultipleDatetimePicker`,
- `TimePicker`,
- `DateRangeSlider`,
- `DateSlider`,
- `DatetimeRangeSlider`,
- `Div`,
- `Dropdown`,
- `FileInput`,
- `MultiChoice`,
- `MultiSelect`,
- `NumericInput`,
- `Paragraph`,
- `PasswordInput`,
- `PreText`,
- `RadioButtonGroup`,
- `RadioGroup`,
- `RangeSlider`,
- `Select`,
- `Slider`,
- `Spinner`,
- `Switch`,
- `Tabs`,
- `TextAreaInput`,
- `TextInput`,
- `Toggle`.

Dodatkowo, zastosuj `HelpButton` z `Tooltip` lub dodaj `Tooltip` do przynajmniej 1 widgetu.

In [ ]:
num_of_events_summer = athlete_events.loc[athlete_events.Season == 'Summer', ['Event', 'Year']].drop_duplicates().groupby(['Year']).count().reset_index()
num_of_events_summer.rename(columns={'Event': 'Count'}, inplace=True)

output_notebook()

p = figure(
    title='Liczba konkurencji na letnich igrzyskach',
    width=800,
    height=600,
    x_axis_label='Rok',
    y_axis_label='Liczba konkurencji',
)

line = p.line(
    x=num_of_events_summer['Year'],
    y=num_of_events_summer['Count'],
    line_width=2,
    color='blue'
)

points = p.scatter(
    num_of_events_summer['Year'],
    num_of_events_summer['Count'],
    size=6,
    color='red',
    line_color='red'
)

p.title.text_color = '#333333'
p.title.align = 'center'
p.background_fill_color = '#f5f5f5'
p.background_fill_alpha = 0.8
p.grid.grid_line_color = '#dddddd'
p.grid.grid_line_alpha = 0.7

picker = ColorPicker(title='Kolor linii')
picker.js_link('color', line.glyph, 'line_color')

years = sorted(num_of_events_summer['Year'].astype(int).unique().tolist())
shared_x_range = Range1d(start=min(years), end=max(years))
range_slider = RangeSlider(start=min(years) - 2, end=max(years) + 2, value=(min(years), max(years)), step=1, title="Zakres lat")
range_slider.js_on_change('value', CustomJS(args=dict(x_range=p.x_range), code="""
    const [start, end] = cb_obj.value;
    x_range.start = start;
    x_range.end = end;
"""))

spinner = Spinner(title='Wielkość punktów', low=5, high=20, step=0.5, value=5, width=80)
spinner.js_link('value', points.glyph, 'size')

show(column(p, row(picker, range_slider, spinner)))